In [ ]:
import joblib
import numpy as np
import pandas as pd

# Load train-test data
X_train, X_test, y_train, y_test = joblib.load("Data/train_test_split.pkl")

print("✅ Data loaded successfully!")
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")


In [ ]:
import os
os.chdir(r"C:\Users\aadit\Documents\Codes\Music Genre identification")
print("✅ Directory changed to:", os.getcwd())


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Train baseline SVM
svm_model = SVC(kernel='rbf', C=10, gamma='scale')  # RBF kernel works well for this task
svm_model.fit(X_train, y_train)

# Evaluate on test data
y_pred = svm_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"🎯 SVM Test Accuracy: {acc*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
import numpy as np

X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn  = np.expand_dims(X_test, axis=2)

print("X_train_cnn shape:", X_train_cnn.shape)
print("X_test_cnn  shape:", X_test_cnn.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Build the 1D CNN model
model = models.Sequential([
    layers.Conv1D(128, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    
    layers.Conv1D(256, kernel_size=3, activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')  # 10 genres
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train_cnn, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
test_loss, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"🎯 1D CNN Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
# -----------------------------------------------
# 🎵 Improved Step 4C: 1D CNN Model (Higher Accuracy Version)
# -----------------------------------------------

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reshape input data
X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn  = np.expand_dims(X_test, axis=2)
print("✅ Data reshaped for CNN:", X_train_cnn.shape)

# ---------------------------------------------------------
# 1️⃣ Improved 1D CNN Architecture
# ---------------------------------------------------------
model = models.Sequential([
    # Conv Block 1
    layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    # Conv Block 2
    layers.Conv1D(256, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    # Conv Block 3 (deeper learning)
    layers.Conv1D(512, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),

    # Fully Connected Layers
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output Layer
    layers.Dense(10, activation='softmax')
])

# ---------------------------------------------------------
# 2️⃣ Compile with Lower Learning Rate
# ---------------------------------------------------------
optimizer = optimizers.Adam(learning_rate=1e-4)  # smaller LR helps deeper models converge

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 3️⃣ Callbacks (Early Stop + LR Reduction)
# ---------------------------------------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ---------------------------------------------------------
# 4️⃣ Train the Improved CNN
# ---------------------------------------------------------
history = model.fit(
    X_train_cnn, y_train,
    epochs=60,             # allow more epochs since early stopping handles overfit
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ---------------------------------------------------------
# 5️⃣ Save Model
# ---------------------------------------------------------
model.save("models/cnn_1d_genre_classifier_v2.h5")
print("✅ Improved 1D CNN model trained and saved as models/cnn_1d_genre_classifier_v2.h5")


In [ ]:
# -----------------------------------------------
# 🎵 Step 4C: 1D CNN Training (No Early Stopping)
# -----------------------------------------------

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Reshape data
X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn  = np.expand_dims(X_test, axis=2)

print("✅ Data reshaped:", X_train_cnn.shape)

# ---------------------------------------------------------
# Build Deeper 1D CNN
# ---------------------------------------------------------
model = models.Sequential([
    layers.Conv1D(128, 5, padding='same', activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(256, 5, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(512, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(10, activation='softmax')
])

optimizer = optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ---------------------------------------------------------
# Use LR Scheduler (Optional)
# ---------------------------------------------------------
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# ---------------------------------------------------------
# Train for Fixed Epochs (No Early Stopping)
# ---------------------------------------------------------
history = model.fit(
    X_train_cnn, y_train,
    epochs=50,               # run all 50 epochs
    batch_size=32,
    validation_split=0.2,
    callbacks=[reduce_lr],
    verbose=1
)

# ---------------------------------------------------------
# Save Model
# ---------------------------------------------------------
model.save("models/cnn_1d_genre_classifier_v3.h5")
print("✅ 1D CNN model trained (no early stopping) and saved as models/cnn_1d_genre_classifier_v3.h5")


In [ ]:
test_loss, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"🎯 1D CNN Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
# ----------------------------------------------------------
# 🎵 Improved 1D CNN (v4C+): Regularized, Tuned, LeakyReLU
# ----------------------------------------------------------

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Reshape input data (keep same)
X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn  = np.expand_dims(X_test, axis=2)
print("✅ Data reshaped for CNN:", X_train_cnn.shape)

# ----------------------------------------------------------
# 1️⃣ Build the Optimized 1D CNN Architecture
# ----------------------------------------------------------
model = models.Sequential([
    # Conv Block 1
    layers.Conv1D(64, kernel_size=3, padding='same',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.LeakyReLU(alpha=0.1),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    # Conv Block 2
    layers.Conv1D(128, kernel_size=3, padding='same',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.LeakyReLU(alpha=0.1),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    # Conv Block 3
    layers.Conv1D(256, kernel_size=3, padding='same',
                  kernel_regularizer=regularizers.l2(1e-4)),
    layers.LeakyReLU(alpha=0.1),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),

    # Dense Layers
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.3),

    # Output Layer
    layers.Dense(10, activation='softmax')
])

# ----------------------------------------------------------
# 2️⃣ Compile with Tuned Optimizer
# ----------------------------------------------------------
optimizer = optimizers.Adam(learning_rate=3e-4)

model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ----------------------------------------------------------
# 3️⃣ Learning Rate Scheduler
# ----------------------------------------------------------
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ----------------------------------------------------------
# 4️⃣ Train Longer (to allow full convergence)
# ----------------------------------------------------------
history = model.fit(
    X_train_cnn, y_train,
    epochs=70,
    batch_size=32,
    validation_split=0.2,
    callbacks=[reduce_lr],
    verbose=1
)

# ----------------------------------------------------------
# 5️⃣ Save Model
# ----------------------------------------------------------
model.save("models/cnn_1d_genre_classifier_v4c_plus.h5")
print("✅ 1D CNN (v4C+) model trained and saved successfully!")


In [ ]:
test_loss, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"🎯 Final Improved 1D CNN Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, max_depth=20)
rf.fit(X_train, y_train)
print(rf.score(X_test, y_test))


In [ ]:
print("Train acc:", history.history['accuracy'][-1])
print("Val acc:", history.history['val_accuracy'][-1])


In [ ]:
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import ReduceLROnPlateau

def residual_block(x, filters, kernel_size=3):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([shortcut, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    return x

X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn = np.expand_dims(X_test, axis=2)

input_shape = (X_train_cnn.shape[1], 1)
inputs = layers.Input(shape=input_shape)

x = layers.Conv1D(64, 3, padding='same', activation='relu')(inputs)
x = residual_block(x, 64)
x = residual_block(x, 128)
x = residual_block(x, 256)
x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
optimizer = optimizers.Adam(learning_rate=3e-4)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_cnn, y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.2,
    callbacks=[reduce_lr],
    verbose=1
)


In [ ]:
# ======================================================
#  🎶 CRNN-based Music Genre Classification (PyTorch)
# ======================================================

import os
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =====================
# CONFIG
# =====================
DATA_DIR = "path_to_your_audio_files"  # <-- change this to your dataset path
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
DURATION = 5.0   # seconds
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# =====================
# HELPER FUNCTIONS
# =====================
def spec_augment(mel):
    """Simple SpecAugment for mel-spectrogram"""
    n_mels, time_steps = mel.shape
    mel = mel.copy()
    for _ in range(2):  # 2 time masks
        t = np.random.randint(0, max(1, time_steps - 20))
        mel[:, t:t+15] = 0
    for _ in range(2):  # 2 freq masks
        f = np.random.randint(0, max(1, n_mels - 8))
        mel[f:f+8, :] = 0
    return mel

# =====================
# DATASET CLASS
# =====================
class AudioDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]

        # safely load audio
        try:
            y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            y = np.zeros(int(SAMPLE_RATE * DURATION))
            sr = SAMPLE_RATE

        # pad or trim
        if len(y) < int(sr * DURATION):
            y = np.pad(y, (0, int(sr * DURATION) - len(y)), 'constant')
        else:
            y = y[:int(sr * DURATION)]

        # augmentations (time, pitch, noise)
        if self.augment:
            if np.random.rand() < 0.3:
                y = librosa.effects.pitch_shift(y, sr, np.random.randint(-2, 3))
            if np.random.rand() < 0.3:
                y = librosa.effects.time_stretch(y, np.random.uniform(0.9, 1.1))
            if np.random.rand() < 0.2:
                noise = np.random.randn(len(y))
                y = y + 0.005 * noise

        # mel-spectrogram
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        # optional SpecAugment
        if self.augment and np.random.rand() < 0.5:
            mel_norm = spec_augment(mel_norm)

        mel_tensor = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0)  # (1, n_mels, time)
        return mel_tensor, torch.tensor(label, dtype=torch.long)

# =====================
# CRNN MODEL
# =====================
class CRNNGenreClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # --- Convolutional feature extractor ---
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )

        # --- Recurrent layer ---
        self.gru = nn.GRU(
            input_size=128, hidden_size=128, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )

        # --- Fully connected head ---
        self.fc = nn.Sequential(
            nn.Linear(128 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: (B, 1, n_mels, time)
        x = self.cnn(x)             # -> (B, 128, n_mels/8, time/8)
        x = x.mean(dim=2)           # average over frequency -> (B, 128, time/8)
        x = x.permute(0, 2, 1)      # -> (B, time, features)
        out, _ = self.gru(x)        # -> (B, time, hidden*2)
        out = out.mean(dim=1)       # global average pooling over time
        return self.fc(out)

# =====================
# DATA PREPARATION
# =====================
def prepare_data(data_dir):
    genres = sorted(os.listdir(data_dir))
    file_paths, labels = [], []
    for idx, genre in enumerate(genres):
        genre_path = os.path.join(data_dir, genre)
        if not os.path.isdir(genre_path): continue
        for file in os.listdir(genre_path):
            if file.endswith((".wav", ".mp3")):
                file_paths.append(os.path.join(genre_path, file))
                labels.append(idx)
    return file_paths, labels, genres

file_paths, labels, genre_names = prepare_data(DATA_DIR)
print(f"Found {len(file_paths)} audio files across {len(genre_names)} genres")

X_train, X_val, y_train, y_val = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

train_dataset = AudioDataset(X_train, y_train, augment=True)
val_dataset   = AudioDataset(X_val, y_val, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

# =====================
# TRAINING LOOP
# =====================
model = CRNNGenreClassifier(num_classes=len(genre_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
EPOCHS = 40
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0

    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(y).sum().item()
        total += y.size(0)

    train_acc = correct / total

    # validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            outputs = model(X)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(y).sum().item()
            total += y.size(0)
    val_acc = correct / total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_crnn_genre.pth")

print("Training finished. Best validation accuracy:", best_val_acc)
